# Task 1, Exercise 2

<p align="center">
  <img src="../src/resources/img/a1_u2_img.png" alt="Schwingungssystem" width="500">
</p>

The depicted oscillatory system, consisting of a homogeneous cylinder (radius $r$, mass $m$) and a spring-damper element (spring stiffness $c$, damping constant $d$), rolls without slipping on a plane. A harmonic force $F(t)=\hat{F}cos⁡\Omega t$ is applied at the center of the cylinder.

### Tasks
1. Isolate the body in its displaced position and indicate all forces and moments acting on it.

2. Derive the equation of motion for the system. Use the general form: $\ddot{q}+2\delta \dot{q} + \omega_{0}^{2}q = b_{0}u+b_{1}\dot{u} + b_{2} \ddot{u}$

3. Determine the amplitude response $V(\Omega)$ and the phase response $\Phi(\Omega)$ of the system. Sketch the amplitude response.

Given: $r,m,c,d,F(t)=\hat{F}cos(\Omega⁡ t)$


Related topics: 
- [Undamped Seismic Harmonic Input](Undamped_Response_to_Harmonic_Seismic_Inputs.ipynb)
- [Undamped Response to Harmonic Direct-Force Inputs](Undamped_Response_to_Harmonic_Direct_Force_Inputs.ipynb)
- [Mass Spring Damper - Direct Force](Mass_Spring_Damper_Direct_Force.ipynb)
- [Transfer Function for Undamped Harmonic Seismic Inputs](Transfer_Function_for_Undamped_Harmonic_Seismic_Inputs.ipynb)
- [Frequency Response to Harmonic Direct-Force Inputs](Frequency_Response_to_Harmonic_Direct_Force_Inputs.ipynb)



In [1]:
%matplotlib widget
%load_ext autoreload
%autoreload 2
import os, sys
dir2 = os.path.abspath('')
dir1 = os.path.dirname(dir2)
if not dir1 in sys.path: sys.path.append(dir1)

import warnings
warnings.filterwarnings("ignore", message="findfont: Font family")

In [2]:
import math 
START_DEFLECTION = 0.1 # min = 0, max = math.pi/5
START_VELOCITY = 0  # min = 0, max = 5
MASS = 0.5 # min = 0.1, max = 5
DEFAULT_C = 30 # min = 10, max = 100
DEFAULT_D = 0.1  # min = 0.1, max = 5
F_HAT = 1 # min = 0.5, max = 1.5
ALPHA = 0.5 
OMEGA = 6.3 

In [ ]:
import numpy as np
from scipy.integrate import odeint

from src.project.calculations.int_solver import validate_solution

correct = False


class StudentSolver:
    def __init__(self) -> None:
        return None

    # Dauerlauf
    def state_space_settled(self, z, t, d, m, c, omega, f_hat):
        [x, x_p] = z  
        delta = d / (3 * m)
        omega_0 = np.sqrt(2 * c / (3 * m))  
        b0 = 2 / (3 * m)
        z_p = [
            x_p,
            -2 * delta * x_p - omega_0**2 * x + b0 * f_hat * np.cos(omega * t),
        ]
        return z_p

    # Hochlauf
    def state_space_accelerated(self, z, t, d, m, c, f_hat, alpha):
        [x, x_p] = z
        delta = d / (3 * m)
        omega_0 = np.sqrt(2 * c / (3 * m))
        b0 = 2 / (3 * m)
        z_p = [
            x_p,
            # write your code here⬇️
            None,
        ]
        return z_p

    def integrate(self, func, start_deflection, start_velocity, t, *args):
        y0 = (start_deflection, start_velocity)

        x = odeint(func=func, y0=y0, t=t, args=args)[:, 0]
        return x 


from src.project.calculations.int_solver import IntSolverAufgabe1Uebung2 as IntSolverCorrect

np.set_printoptions(precision=4, suppress=True, linewidth=120)
student_solver = StudentSolver()
correct_solver = IntSolverCorrect()
t = np.linspace(0, 20, 500)


# settled
settled_student = student_solver.integrate(
                                student_solver.state_space_settled, 
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                DEFAULT_D,
                                MASS,
                                DEFAULT_C,
                                OMEGA, 
                                F_HAT)

settled_correct = correct_solver.integrate(
                                correct_solver.state_space_settled, 
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                DEFAULT_D,
                                MASS,
                                DEFAULT_C,
                                OMEGA, 
                                F_HAT)

# accelerated
accelerated_student = student_solver.integrate(
                                student_solver.state_space_accelerated, 
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                DEFAULT_D, MASS, DEFAULT_C, F_HAT, ALPHA)
accelerated_correct = correct_solver.integrate(
                                correct_solver.state_space_accelerated,
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                DEFAULT_D,
                                MASS,
                                DEFAULT_C,
                                F_HAT, 
                                ALPHA)


# test results
print("Settled: ")
correct_settled = validate_solution(settled_correct, settled_student, relative_threshold=0.05)
print()
print("Accelerated: ")
correct_acc = validate_solution(accelerated_correct, accelerated_student, relative_threshold=0.05)

# solution is correct if both settled and accelerated are correct
correct = correct_settled and correct_acc



In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

from src.project.gui.gui_a1_u2 import GUI
from src.project.anim.animations.a1_u2_animation import Aufgabe1

if correct: 
    animation_instance = Aufgabe1(calculator=student_solver)
    app = GUI(default_c=DEFAULT_C,
            default_d=DEFAULT_D,
            animation_instance=animation_instance)
    display(app.app_layout)
else: 
    print("Your solution is not correct. Please check your implementation.")